# `config.ipynb` — Parametrización y credenciales

Equivalente en notebook a **`config.py`**. Aquí, y solo aquí, se definen:

- Las rutas del proyecto (con `os`, sin nada absoluto "hardcodeado").
- Las credenciales/parámetros de conexión a **SQL Server** (origen) y
  **PostgreSQL** (destino), leídos con `os.environ.get(VARIABLE, valor_por_defecto)`.
- Los parámetros de negocio del enunciado (tasa USD→EUR, catálogo de
  categorías de moda, regex de validación de DNI).

Este notebook **no ejecuta ninguna lógica de negocio**: solo declara variables.
`transform.ipynb` y `etl_main.ipynb` lo cargan al principio con:

```python
%run config.ipynb
```

`%run` es, en un notebook, el equivalente a un `from config import *` de un
script `.py`: ejecuta este fichero y todas sus variables (`DIR_PROYECTO`,
`SQL_SERVER`, `PG_HOST`, `TASA_USD_EUR`...) quedan disponibles en el notebook
que lo invoca. Por eso puede vivir como notebook independiente y, aun así,
"importarse" desde los otros dos.


## Librerías

Solo se necesita `os` (rutas y variables de entorno) y `re` (para compilar
la expresión regular de validación de DNI, que se usa en `transform.ipynb`).

In [ ]:
import os
import re

print('Librerías de configuración importadas correctamente')


## Rutas del proyecto

Portable a cualquier equipo: nada de rutas absolutas "hardcodeadas". Se parte
de `os.getcwd()` y, si el notebook se ejecuta desde `src/` (comportamiento
por defecto de Jupyter, que sitúa el directorio de trabajo en la carpeta del
notebook), se sube un nivel para situar la raíz real del proyecto.

In [ ]:
# --- Rutas del proyecto (portable a cualquier equipo, sin rutas absolutas) ---
# El notebook vive en .../<PROYECTO>/src/. Si se ejecuta desde ahí (comportamiento
# por defecto de Jupyter), subimos un nivel para situar la raíz del proyecto.
DIR_SRC = os.getcwd()
DIR_PROYECTO = (
    os.path.abspath(os.path.join(DIR_SRC, os.pardir))
    if os.path.basename(DIR_SRC) == 'src'
    else DIR_SRC
)

DIR_SQL          = os.path.join(DIR_PROYECTO, 'sql')
DIR_LOGS         = os.path.join(DIR_PROYECTO, 'logs')
DIR_RESULTADOS   = os.path.join(DIR_PROYECTO, 'Resultados')
DIR_SAMPLE_DATA  = os.path.join(DIR_PROYECTO, 'sample_data')

RUTA_DDL          = os.path.join(DIR_SQL, 'ddl_destino_postgresql.sql')
RUTA_LOG          = os.path.join(DIR_LOGS, 'ejecucion_etl.log')
RUTA_ORIGEN_LOCAL = os.path.join(DIR_SAMPLE_DATA, 'origen_local_demo.db')

# --- Conexión origen: SQL Server ---
SQL_DRIVER   = os.environ.get('SQL_DRIVER', '{ODBC Driver 17 for SQL Server}')
SQL_SERVER   = os.environ.get('SQL_SERVER', 'EM2026008667')
SQL_DATABASE = os.environ.get('SQL_DATABASE', 'BASE_DATOS_TIENDA_ROPAS_MARCAS')
# Si se definen SQL_USER/SQL_PASSWORD se usa autenticación SQL; si no, autenticación
# integrada de Windows (Trusted_Connection), igual que en el resto de scripts del equipo.
SQL_USER     = os.environ.get('SQL_USER', '')
SQL_PASSWORD = os.environ.get('SQL_PASSWORD', '')

# --- Conexión destino: PostgreSQL (desplegado vía docker-compose.yml) ---
PG_HOST     = os.environ.get('POSTGRES_HOST', 'localhost')
PG_PORT     = os.environ.get('POSTGRES_PORT', '5432')
PG_DATABASE = os.environ.get('POSTGRES_DB', 'db_fashionshop_dw')
PG_USER     = os.environ.get('POSTGRES_USER', 'admin_etl')
PG_PASSWORD = os.environ.get('POSTGRES_PASSWORD', 'Password123!')

# --- Parámetros de negocio (enunciado RF-01 a RF-06) ---
TASA_USD_EUR = float(os.environ.get('TASA_USD_EUR', '1.15'))  # 1.15 USD = 1.00 EUR
CATEGORIAS_MODA = {
    'Camisetas y Polos', 'Pantalones y Jeans', 'Ropa Deportiva',
    'Calzado y Zapatillas', 'Accesorios y Cinturones',
}  # RF-02: el resto (Hamburguesas, Pizzas, Sándwiches) queda fuera del alcance
RE_DNI = re.compile(r'^\d{8}[A-Za-z]$')  # RF-01: 8 dígitos + 1 letra

print(f'Proyecto  : {DIR_PROYECTO}')
print(f'SQL Server: {SQL_SERVER}/{SQL_DATABASE}')
print(f'PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DATABASE} (usuario: {PG_USER})')


## Confirmación

Un `print` de control para verificar, nada más ejecutar este notebook (solo
o vía `%run`), que la configuración se ha cargado con los valores esperados.

In [ ]:
print('--- config.ipynb cargado correctamente ---')
print(f'Proyecto  : {DIR_PROYECTO}')
print(f'SQL Server: {SQL_SERVER}/{SQL_DATABASE}')
print(f'PostgreSQL: {PG_HOST}:{PG_PORT}/{PG_DATABASE} (usuario: {PG_USER})')
print(f'Tasa USD->EUR: {TASA_USD_EUR} | Categorías de moda: {len(CATEGORIAS_MODA)}')
